In [3]:
from pathlib import Path
import pandas as pd
import os

path = os.path.join(os.getcwd(), "..", "..", "singapore_weather_monthly")

root = Path(path)


In [4]:
root

WindowsPath('c:/Computer Science/Hackathons/huawei/naisc-2025/model/flood/../../singapore_weather_monthly')

In [5]:

# the full schema you expect
EXPECTED_COLS = [
    "timestamp",
    "temperature",
    "rainfall",
    "humidity",
    "wind_direction",
    "wind_speed",
]

station_parts = {}

for month_dir in sorted(root.iterdir()):
    if not month_dir.is_dir():
        continue

    for csv_path in month_dir.glob("*.csv"):
        station = csv_path.stem

        # try reading with header; fall back to no-header
        try:
            df = pd.read_csv(csv_path, parse_dates=["timestamp"])
        except ValueError:
            df = pd.read_csv(
                csv_path,
                header=None,
                names=EXPECTED_COLS,
                parse_dates=[0],
            )

        # normalize column names
        df.columns = [c.lower() for c in df.columns]

        # ensure *all* expected columns are present, filling missing ones with <NA>
        df = df.reindex(
            columns=EXPECTED_COLS,
            fill_value=pd.NA,
        )

        # set timestamp index
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.set_index("timestamp").sort_index()

        station_parts.setdefault(station, []).append(df)

# concatenate per-station
for station, parts in station_parts.items():
    full = pd.concat(parts)
    full = full[~full.index.duplicated()]  # drop any true duplicates
    full = full.sort_index()

    # save or keep in memory
    full.to_csv(f"{station}_all_timeseries.csv")
    station_parts[station] = full


C:\Users\Parvez\AppData\Local\Temp\ipykernel_23340\1417172302.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(
C:\Users\Parvez\AppData\Local\Temp\ipykernel_23340\1417172302.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"])


DateParseError: Unknown datetime string format, unable to parse: S109, at position 1

In [6]:
import os
import glob
import pandas as pd

# 1. Define folder and file pattern
data_folder = os.path.join(os.getcwd(), "..", "..", "singapore_weather_combined") # adjust if needed
csv_pattern = os.path.join(data_folder, '*.csv')

# 2. Define full hourly datetime index
start = '2024-01-01 00:00:00'
end = '2025-04-26 23:00:00'
full_index = pd.date_range(start=start, end=end, freq='H')

# 3. Required columns
required_cols = ['timestamp', 'temperature', 'rainfall', 'humidity', 'wind_direction', 'wind_speed']

# 4. Process each CSV
for filepath in glob.glob(csv_pattern):
    filename = os.path.basename(filepath)
    # Skip metadata or non-station files if necessary
    if filename == 'combined_station_metadata.csv':
        continue

    # Read original CSV
    df = pd.read_csv(filepath, parse_dates=['timestamp'])

    # Ensure timestamp column exists
    if 'timestamp' not in df.columns:
        continue  # or handle appropriately

    # Set timestamp as index
    df = df.set_index('timestamp')

    # Reindex to full hourly range
    df = df.reindex(full_index)

    # Reset index to column
    df = df.reset_index().rename(columns={'index': 'timestamp'})

    # Ensure all required columns exist
    for col in required_cols:
        if col not in df.columns:
            df[col] = pd.NA

    # Reorder columns
    df = df[required_cols]

    # Save standardized CSV
    out_path = os.path.join(data_folder, 'standardized_' + filename)
    df.to_csv(out_path, index=False)

    print(f'Standardized {filename} -> standardized_{filename}')


C:\Users\Parvez\AppData\Local\Temp\ipykernel_23340\3862270781.py:12: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(start=start, end=end, freq='H')


Standardized S06_Paya_Lebar.csv -> standardized_S06_Paya_Lebar.csv
Standardized S07_Lornie_Road.csv -> standardized_S07_Lornie_Road.csv
Standardized S08_Upper_Thomson_Road.csv -> standardized_S08_Upper_Thomson_Road.csv
Standardized S102_Semakau_Landfill.csv -> standardized_S102_Semakau_Landfill.csv
Standardized S104_Woodlands_Avenue_9.csv -> standardized_S104_Woodlands_Avenue_9.csv
Standardized S106_Pulau_Ubin.csv -> standardized_S106_Pulau_Ubin.csv
Standardized S107_East_Coast_Parkway.csv -> standardized_S107_East_Coast_Parkway.csv
Standardized S108_Marina_Gardens_Drive.csv -> standardized_S108_Marina_Gardens_Drive.csv
Standardized S109_Ang_Mo_Kio_Avenue_5.csv -> standardized_S109_Ang_Mo_Kio_Avenue_5.csv
Standardized S111_Scotts_Road.csv -> standardized_S111_Scotts_Road.csv
Standardized S112_Lim_Chu_Kang_Road.csv -> standardized_S112_Lim_Chu_Kang_Road.csv
Standardized S113_Marine_Parade_Road.csv -> standardized_S113_Marine_Parade_Road.csv
Standardized S114_Choa_Chu_Kang_Avenue_4.csv -

In [7]:
import os
import glob
import pandas as pd

# Folder containing standardized CSVs
data_folder = os.path.join(os.getcwd(), "..", "..", "singapore_weather_combined")
pattern = os.path.join(data_folder, 'standardized_*.csv')

# Process each standardized CSV
for filepath in glob.glob(pattern):
    filename = os.path.basename(filepath)
    print(f'Processing {filename}...')

    # Read CSV and parse timestamp
    df = pd.read_csv(filepath, parse_dates=['timestamp'])
    
    # Set timestamp as index
    df = df.set_index('timestamp').sort_index()
    
    # Perform linear interpolation for gaps <= 3 hours
    # limit=4 ensures only runs of NaNs of length up to 3 are filled
    df_interpolated = df.interpolate(
        method='time',
        limit=3,
        limit_direction='both'
    )
    
    # Write output
    out_path = os.path.join(data_folder, 'imputed_' + filename)
    df_interpolated.reset_index().to_csv(out_path, index=False)
    
    print(f'  Saved imputed data to {os.path.basename(out_path)}')


Processing standardized_S06_Paya_Lebar.csv...
  Saved imputed data to imputed_standardized_S06_Paya_Lebar.csv
Processing standardized_S07_Lornie_Road.csv...
  Saved imputed data to imputed_standardized_S07_Lornie_Road.csv
Processing standardized_S08_Upper_Thomson_Road.csv...
  Saved imputed data to imputed_standardized_S08_Upper_Thomson_Road.csv
Processing standardized_S102_Semakau_Landfill.csv...
  Saved imputed data to imputed_standardized_S102_Semakau_Landfill.csv
Processing standardized_S104_Woodlands_Avenue_9.csv...
  Saved imputed data to imputed_standardized_S104_Woodlands_Avenue_9.csv
Processing standardized_S106_Pulau_Ubin.csv...
  Saved imputed data to imputed_standardized_S106_Pulau_Ubin.csv
Processing standardized_S107_East_Coast_Parkway.csv...
  Saved imputed data to imputed_standardized_S107_East_Coast_Parkway.csv
Processing standardized_S108_Marina_Gardens_Drive.csv...
  Saved imputed data to imputed_standardized_S108_Marina_Gardens_Drive.csv
Processing standardized_S109

In [9]:
import os
import glob
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# 1. Load station metadata
meta_path = os.path.join(os.getcwd(), "..", "..", "singapore_weather_combined", "combined_station_metadata.csv")
meta_df = pd.read_csv(meta_path)
meta_df = meta_df.set_index('id')[['latitude', 'longitude']]

station_ids = meta_df.index.tolist()
coords = meta_df.to_dict('index')  # {id: {'latitude':..., 'longitude':...}, ...}

# 2. Compute pairwise distances (meters) via Haversine
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlambda/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

# Build distance map: {id: [(neighbor_id, distance), ...]}
distance_map = {}
for sid in station_ids:
    lat1, lon1 = coords[sid]['latitude'], coords[sid]['longitude']
    distances = [(other, haversine(lat1, lon1, coords[other]['latitude'], coords[other]['longitude']))
                 for other in station_ids if other != sid]
    # Sort ascending by distance
    distance_map[sid] = sorted(distances, key=lambda x: x[1])

# 3. IDW parameters
n_neighbors = 5  # use 5 nearest available stations
power = 2        # inverse-distance-squared weighting

# 4. Process each station CSV for IDW-based imputation
data_folder = os.path.join(os.getcwd(), "..", "..", "singapore_weather_combined")
pattern = os.path.join(data_folder, 'imputed_standardized_*.csv')
features = ['temperature', 'rainfall', 'humidity', 'wind_direction', 'wind_speed']
# Before your IDW loop, build station_dfs like this:
station_dfs = {}
for path in glob.glob('imputed_standardized_S*_*.csv'):
    df = pd.read_csv(path, parse_dates=['timestamp']).set_index('timestamp')
    sid = os.path.basename(path).split('_')[2]  # e.g. 'S109'
    station_dfs[sid] = df

# Now station_dfs['S109'] is the DataFrame for station S109,
# station_dfs['S117'] is for S117, etc.


for filepath in glob.glob(pattern):
    df = pd.read_csv(filepath, parse_dates=['timestamp']).set_index('timestamp')
    sid = os.path.basename(filepath).split('_')[2]  # extract Sxx from filename
    neighbors = distance_map[sid]

    print(f'Imputing for station {sid}...')

    # For each feature and each missing timestamp, perform IDW from neighbors
    for feat in features:
        missing = df[df[feat].isna()].index
        print(f'  Missing timestamps for {feat}: {len(missing)}')
        for t in missing:
            vals, dists = [], []
            # find up to n_neighbors with valid data at time t
            for neighbor_id, dist in neighbors:
                if len(vals) >= n_neighbors:
                    break
                v = station_dfs.get(neighbor_id, pd.DataFrame()).get(feat)
                # Need to load neighbor df once
                if v is None:
                    # lazy-load neighbor df
                    neighbor_file = glob.glob(os.path.join(data_folder, f'imputed_standardized_{neighbor_id}_*.csv'))
                    if neighbor_file:
                        station_dfs[neighbor_id] = pd.read_csv(neighbor_file[0], parse_dates=['timestamp']).set_index('timestamp')
                        v_series = station_dfs[neighbor_id][feat] if feat in station_dfs[neighbor_id] else None
                    else:
                        v_series = None
                else:
                    v_series = v
                if v_series is not None and not pd.isna(v_series.get(t, np.nan)):
                    vals.append(v_series[t])
                    dists.append(dist)
            # Compute IDW if any neighbor values found
            if vals:
                weights = np.array([1/(d**power) for d in dists])
                df.at[t, feat] = np.sum(weights * vals) / np.sum(weights)
    # Save back the imputed file
    out_path = os.path.join(data_folder, 'idw_imputed_standardized_' + os.path.basename(filepath))
    df.reset_index().to_csv(out_path, index=False)
    print(f'  -> Saved {out_path}')


Imputing for station S06...
  -> Saved c:\Computer Science\Hackathons\huawei\naisc-2025\model\flood\..\..\singapore_weather_combined\idw_imputed_standardized_imputed_standardized_S06_Paya_Lebar.csv
Imputing for station S07...
  -> Saved c:\Computer Science\Hackathons\huawei\naisc-2025\model\flood\..\..\singapore_weather_combined\idw_imputed_standardized_imputed_standardized_S07_Lornie_Road.csv
Imputing for station S08...
  -> Saved c:\Computer Science\Hackathons\huawei\naisc-2025\model\flood\..\..\singapore_weather_combined\idw_imputed_standardized_imputed_standardized_S08_Upper_Thomson_Road.csv
Imputing for station S102...
  -> Saved c:\Computer Science\Hackathons\huawei\naisc-2025\model\flood\..\..\singapore_weather_combined\idw_imputed_standardized_imputed_standardized_S102_Semakau_Landfill.csv
Imputing for station S104...
  -> Saved c:\Computer Science\Hackathons\huawei\naisc-2025\model\flood\..\..\singapore_weather_combined\idw_imputed_standardized_imputed_standardized_S104_Woodla

KeyboardInterrupt: 

Change wind speed and dir to direction

In [128]:
n_neighbors = 5

pattern = os.path.join(data_folder, 'imputed_standardized_S*_*.csv')
# 1. Load station metadata
meta_path = os.path.join(os.getcwd(), "..", "..", "singapore_weather_combined", "combined_station_metadata.csv")
meta_df = pd.read_csv(meta_path)
meta_df = meta_df.set_index('id')[['latitude', 'longitude']]

coords = meta_df.to_dict('index')  # {id: {'latitude':..., 'longitude':...}, ...}
station_dfs = {}
for path in glob.glob(pattern):
    df = pd.read_csv(path, parse_dates=['timestamp']).set_index('timestamp')
    sid = os.path.basename(path).split('_')[2]  # e.g. 'S109'
    station_dfs[sid] = df
for df in station_dfs.values():
    # assuming df has columns 'wind_speed' and 'wind_direction'
    theta = np.deg2rad(df['wind_direction'])
    df['wind_u'] = df['wind_speed'] * np.sin(theta)
    df['wind_v'] = df['wind_speed'] * np.cos(theta)
    # then drop the old columns
    df.drop(columns=['wind_direction', 'wind_speed'], inplace=True)
m, T, F = 70, 11568, 5 
features = ['temperature','rainfall','humidity','wind_u','wind_v']
import pandas as pd

# 1. Load your metadata
meta = pd.read_csv(meta_path)

# 2. Extract the list of station IDs in a consistent order
#    (here we filter for IDs that start with “S” and preserve file order)
station_ids = [sid for sid in meta['id'].tolist() if sid.startswith('S')]

# 3. Build the index mapping
station_index = { sid: idx for idx, sid in enumerate(station_ids) }

# 4. Example: look up the index for station “S109”
print(station_index['S109'])  # e.g. 0 if it’s first in station_ids
# station_ids = list of all Sxx
# times = the common hourly index
# features = ['temperature','rainfall','humidity','wind_direction','wind_speed']
arr = np.full((m, T, F), np.nan)   # m stations, T timestamps, F features
for sid, df in station_dfs.items():
    i = station_index[sid]
    arr[i,:,:] = df[features].values

# coords[sid] = (lat,lon) for each station
dist = np.full((m,m), np.inf) 
for i,si in enumerate(station_ids):
  for j,sj in enumerate(station_ids):
    if i!=j:
      dist[i,j] = haversine(coords[si]["latitude"], coords[si]["longitude"], coords[sj]["latitude"], coords[sj]["longitude"])
W_glob = 1.0 / (dist ** power)
W_glob /= W_glob.sum(axis=1, keepdims=True)  # normalize weights                       # store in add


0


In [129]:
import numpy as np

# Assume arr of shape (m, T, F) and W_glob of shape (m, m) are already defined
# m = number of stations, T = number of time steps, F = number of features

# 1. Build mask of missing entries
mask = np.isnan(arr)  # shape (m, T, F), True where arr is NaN

# 2. Prepare add array to hold imputed values
add = np.zeros_like(arr)  # same shape

# 3. For each feature, compute the global IDW filled values
for f in range(arr.shape[2]):
    feat = arr[:, :, f]  # (m, T)
    # Numerator: weighted sum, treating NaN as zero
    num = W_glob.dot(np.nan_to_num(feat, nan=0.0))  # (m, T)
    # Denominator: sum of weights where data exists
    valid = (~np.isnan(feat)).astype(float)         # (m, T)
    den = W_glob.dot(valid)                         # (m, T)
    den[den == 0] = np.nan
    filled = num / den                              # (m, T)
    add[:, :, f] = filled                           # store in add

# 4. Compute the completed (fully imputed) array
completed = np.array(arr, copy=True)  # make a copy
completed[mask] = add[mask]           # fill only where arr was NaN

# Now `add` contains the imputed values for every position,
# `mask` marks where arr was originally missing,
# and `completed` is the final filled array.
